In [1]:
!pip install  --upgrade "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 18.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


# Encoder Classifier

In [4]:
# Unzim the file
!unzip  /content/genre_classifier.zip -d /content/

Archive:  /content/genre_classifier.zip
   creating: /content/genre_classifier/
   creating: /content/genre_classifier/checkpoint-332/
  inflating: /content/genre_classifier/checkpoint-332/adapter_config.json  
  inflating: /content/genre_classifier/checkpoint-332/optimizer.pt  
  inflating: /content/genre_classifier/checkpoint-332/rng_state.pth  
  inflating: /content/genre_classifier/checkpoint-332/trainer_state.json  
  inflating: /content/genre_classifier/checkpoint-332/adapter_model.safetensors  
  inflating: /content/genre_classifier/checkpoint-332/scaler.pt  
  inflating: /content/genre_classifier/checkpoint-332/tokenizer_config.json  
  inflating: /content/genre_classifier/checkpoint-332/scheduler.pt  
  inflating: /content/genre_classifier/checkpoint-332/training_args.bin  
  inflating: /content/genre_classifier/checkpoint-332/README.md  
  inflating: /content/genre_classifier/checkpoint-332/tokenizer.json  
   creating: /content/genre_classifier/checkpoint-1328/
  inflating: 

In [5]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset, DatasetDict
from sklearn.model_selection import train_test_split
from peft import PeftModel
import numpy as np


## Load the dataset
stories = load_dataset('FareedKhan/1k_stories_100_genre')

print("Original dataset")
print(stories)
print(100*"_")

genres = sorted(set(stories['train']['genre']))
id2lab = {i:genres[i] for i in range(len(genres))}
lab2id = {genres[i]:i for i in range(len(genres))}

labels = [lab2id[genre] for genre in stories["train"]["genre"]]

## Split the data
# Create indices
indices = list(range(len(stories["train"])))

# First split: 80% train, 20% temp
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,
    random_state=42,
)


# Second split: temp -> 10% val, 10% test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=[labels[i] for i in temp_idx],
    random_state=42,
)

# Create a DatasetDict of the raw splits
raw_splits = DatasetDict({
    "train": stories["train"].select(train_idx),
    "val": stories["train"].select(val_idx),
    "test": stories["train"].select(test_idx)
})
print("Split dataset")
print(raw_splits)
print(100*"_")


## Load the model

# Explicitly set the device to CPU
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# Load the tokenizer and base model
base_model_id = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Initialize base model with the correct number of classes
base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_id,
    num_labels=len(genres), # num_labels should be len(genres) (99)
    id2label=id2lab,
    label2id=lab2id,
)

# Attach trained LoRA adapter
adapter_path = "/content/genre_classifier"
model = PeftModel.from_pretrained(base_model, adapter_path)

# Move the combined model to CPU and set to evaluation mode
model.to(device)
model.eval() #  models instantiated via from_pretrained() are already set to evaluation mode by default, but explicitly calling it is good

README.md:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

1k_stories_100_genre.csv:   0%|          | 0.00/5.92M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Original dataset
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'story', 'genre'],
        num_rows: 1000
    })
})
____________________________________________________________________________________________________
Split dataset
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'story', 'genre'],
        num_rows: 800
    })
    val: Dataset({
        features: ['id', 'title', 'story', 'genre'],
        num_rows: 100
    })
    test: Dataset({
        features: ['id', 'title', 'story', 'genre'],
        num_rows: 100
    })
})
____________________________________________________________________________________________________
cpu


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): DistilBertForSequenceClassification(
      (distilbert): DistilBertModel(
        (embeddings): Embeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (transformer): Transformer(
          (layer): ModuleList(
            (0-5): 6 x TransformerBlock(
              (attention): DistilBertSelfAttention(
                (q_lin): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  )
      

In [18]:
def classify_story(text):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Move inputs to CPU (though they default to CPU, this is good practice)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Disable gradient tracking for efficient inference
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

        # Get the predicted class index
        predicted_class_id = torch.argmax(logits, dim=-1).item()

    return model.config.id2label[predicted_class_id]

In [21]:
for i, (story, gen) in enumerate(zip(raw_splits['test']['story'], raw_splits['test']['genre'])):
  pred_label = classify_story(story)
  print(f"Predicted Genre: {pred_label}")
  print(f"Actual Genre:    {gen}")
  print(50*"_")
  if i == 10:
    break

Predicted Genre: Coming-of-Age
Actual Genre:    Coming-of-Age
__________________________________________________
Predicted Genre: Legal Drama
Actual Genre:    Legal Drama
__________________________________________________
Predicted Genre: War
Actual Genre:    War
__________________________________________________
Predicted Genre: Biographical
Actual Genre:    Evolutionary Fiction
__________________________________________________
Predicted Genre: Artificial Intelligence
Actual Genre:    Artificial Intelligence
__________________________________________________
Predicted Genre: Science Fiction
Actual Genre:    Hard Science Fiction
__________________________________________________
Predicted Genre: Artistic Drama
Actual Genre:    Coming-of-Middle-Age
__________________________________________________
Predicted Genre: Medical Drama
Actual Genre:    Medical Drama
__________________________________________________
Predicted Genre: Artistic Drama
Actual Genre:    Inspirational
______________